# Jeux répétés : le tournoi et la fragilité de la coopération

**Objectifs** : calculer un équilibre répété et comparer huit stratégies en tournoi.

**Prérequis** : dilemme du prisonnier, espérance et C#. **Durée** : 75 minutes.

**Navigation** : [Version Python](GameTheory-14b-Jeux-Repetes-Tournoi-Axelrod-Python.ipynb) | [← 14 — Jeux différentiels](GameTheory-14-DifferentialGames-Python.ipynb) · [↑ Série GameTheory](README.md) · [15 — Jeux coopératifs →](GameTheory-15-CooperativeGames-Python.ipynb)


Le dilemme du prisonnier **épisode unique** a un équilibre strict : la défection commune. Le palier [02](GameTheory-02-NormalForm-Python.ipynb) l'a éliminée par stratégies dominées, le palier [04](GameTheory-04-NashEquilibrium-Python.ipynb) l'a calculée. Ce carnet répond à la question naturelle qui suit : **que se passe-t-il quand le jeu se répète ?**

**Position dans la série.** Le sujet existe déjà sous deux angles : [06](GameTheory-06-EvolutionTrust-Python.ipynb) le traite de façon **évolutionnaire** (tournois, dynamique du réplicateur, librairie Axelrod), [Dynamics-01](Dynamics/Dynamics-01-Jeux-Repetes-Folk-Theorem-Python.ipynb) démontre le théorème folk et [Dynamics-02](Dynamics/Dynamics-02-Jeux-Repetes-Lean-Python.ipynb) en prouve le seuil critique en Lean. Ce carnet isole le **mécanisme d'équilibre** et le met au banc, en C# et sans dépendance externe : pourquoi l'horizon fini dénoue la coopération, quel seuil de patience exact la rétablit, et ce que le bruit fait au classement — chaque affirmation mesurée dans les sorties :

1. **Horizon fini** : la récursion arrière dénoue la coopération — si le nombre de tours est connu, l'équilibre parfait en sous-jeux est la défection dès le premier coup.
2. **Horizon infini (escompté)** : la coopération devient soutenable — on calcule le seuil exact du facteur d'escompte au-dessus duquel la stratégie *rancunier* rend la défection non profitable, et on le vérifie numériquement de part et d'autre.
3. **Tournoi** : huit stratégies s'affrontent en round-robin, puis le même tournoi est rejoué avec du **bruit d'exécution** multi-graine — la stratégie gagnante sans bruit s'effondre quand les coups sont mal transmis, et l'effondrement est mesuré, pas raconté.

Tout ce carnet est **auto-contenu** : moteur de match, stratégies et mesures sont écrits ici, en C# avec la BCL .NET pour les tirages aléatoires. Chaque sortie est celle de notre simulation ; aucune valeur n'est recopiée d'ailleurs.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Globalization;
CultureInfo.DefaultThreadCurrentCulture=CultureInfo.InvariantCulture;
Console.WriteLine($"Runtime : {System.Runtime.InteropServices.RuntimeInformation.FrameworkDescription}");
int T=5,R=3,P=1,S=0;
(int,int) Gain(char a,char b) => (a,b) switch {('C','C')=>(R,R),('C','D')=>(S,T),('D','C')=>(T,S),_=>(P,P)};
foreach(char a in "CD") foreach(char b in "CD") Console.WriteLine($"{a},{b}: {Gain(a,b)}");
foreach(char b in "CD") Console.WriteLine($"contre {b} : C={Gain('C',b).Item1}, D={Gain('D',b).Item1}");

Runtime : .NET 10.0.14


C,C: (3, 3)


C,D: (0, 5)


D,C: (5, 0)


D,D: (1, 1)


contre C : C=3, D=5


contre D : C=0, D=1


La défection **domine strictement** la coopération : contre un adversaire qui coopère, D rapporte $T=5 > R=3$ ; contre un adversaire qui fait défection, D rapporte $P=1 > S=0$. L'équilibre de l'épisode unique est donc $(D, D)$ avec gain $1$ chacun — alors que $(C, C)$ rapporterait $3$ chacun. C'est exactement le paradoxe : l'équilibre est dominé, au sens commun, par un profil non credible en stratégies dominantes. La répétition est le mécanisme qui rend la coopération **credible**.

## 2. Horizon fini : le dénouement par récursion arrière

Supposons le jeu répété $T$ fois, $T$ connu des deux joueurs. On calcule l'équilibre parfait en sous-jeux par induction **depuis le dernier tour**. À chaque tour restant, on demande à chaque joueur sa meilleure réponse, la valeur de continuation étant déjà connue — et comme les gains des tours restants ne dépendent pas de l'action courante autrement que par cette valeur, la meilleure réponse au tour $k$ est celle de l'épisode unique.

In [2]:
char Meilleure1(Dictionary<(char,char),double> suite,char adversaire)=>suite[('C',adversaire)]>suite[('D',adversaire)]?'C':'D';
char Meilleure2(Dictionary<(char,char),double> suite,char adversaire)=>suite[(adversaire,'C')]>suite[(adversaire,'D')]?'C':'D';
(char,char) Equilibre(Dictionary<(char,char),double> continuation){
 var s1=new Dictionary<(char,char),double>();var s2=new Dictionary<(char,char),double>();
 foreach(char a in "CD")foreach(char b in "CD"){s1[(a,b)]=Gain(a,b).Item1+continuation[(a,b)];s2[(a,b)]=Gain(a,b).Item2+continuation[(a,b)];}
 foreach(char a in "CD")foreach(char b in "CD")if(Meilleure1(s1,b)==a&&Meilleure2(s2,a)==b)return(a,b);
 throw new Exception("Aucun équilibre");
}
foreach(int horizon in new[]{1,2,3,5,10}){var suite=new Dictionary<(char,char),double>();foreach(char a in "CD")foreach(char b in "CD")suite[(a,b)]=0; (char,char) eq=('C','C');for(int i=0;i<horizon;i++){eq=Equilibre(suite);foreach(char a in "CD")foreach(char b in "CD")suite[(a,b)]+=Gain(eq.Item1,eq.Item2).Item1;}Console.WriteLine($"horizon {horizon} : premier profil={eq}, valeur={suite[eq]:F1}");}

horizon 1 : premier profil=(D, D), valeur=1.0


horizon 2 : premier profil=(D, D), valeur=2.0


horizon 3 : premier profil=(D, D), valeur=3.0


horizon 5 : premier profil=(D, D), valeur=5.0


horizon 10 : premier profil=(D, D), valeur=10.0


Quel que soit l'horizon fini testé — 1, 2, 3, 5 ou 10 — le profil d'équilibre parfait au **premier** tour est $(D, D)$. C'est le dénouement (*unraveling*) : au dernier tour, aucune menace n'a de valeur (il n'y a plus de suite), donc chacun fait défection ; sachant cela, l'avant-dernier tour n'a plus rien à protéger, donc défection ; et ainsi de suite jusqu'au premier coup. **Connaître la fin du jeu détruit la coopération.** La sortie le montre sur cinq horizons : la valeur totale croît linéairement avec l'horizon (l'horizon multiplié par le gain de punition $P = 1$ : 1, 2, 3, 5, 10), et le gain de coopération $R$ n'apparaît jamais.

## 3. Horizon infini : le seuil d'escompte qui soutient la coopération

Si le jeu se répète sans fin connue, avec un facteur d'escompte $\delta \in [0,1)$ (un gain dans $k$ tours vaut $\delta^k$ fois son montant), la menace redevient crédible. Considérons la stratégie **rancunier** (*grim trigger*) : coopérer tant que personne n'a fait défection, sinon défection pour toujours.

Sur le chemin de coopération, dévier une fois rapporte $T$ puis $P$ à jamais ; rester coopératif rapporte $R$ à jamais. La déviation est non profitable si :

$$T + \sum_{k \geq 1} \delta^k P \;\leq\; \sum_{k \geq 0} \delta^k R
\quad\Longleftrightarrow\quad
\delta \;\geq\; \frac{T - R}{T - P} = \delta^\star$$

Le calcul ci-dessous mesure $\delta^\star$ pour notre matrice, puis **vérifie le seuil numériquement** en comparant les valeurs escomptées tronquées à 500 tours, de part et d'autre du seuil.

In [3]:
double deltaStar=(double)(T-R)/(T-P); int horizon=500;
double ValeurEscomptee(IEnumerable<double> gains,double delta)=>gains.Take(horizon).Select((g,k)=>Math.Pow(delta,k)*g).Sum();
Console.WriteLine($"Seuil théorique : {deltaStar:F4}, horizon de troncature {horizon}");
foreach(double delta in new[]{0.40,0.49,0.50,0.51,0.60}){double coop=ValeurEscomptee(Enumerable.Repeat((double)R,horizon),delta),devi=ValeurEscomptee(new[]{(double)T}.Concat(Enumerable.Repeat((double)P,horizon-1)),delta);double diff=devi-coop;Console.WriteLine($"delta={delta:F2}, coopération={coop:F3}, déviation={devi:F3}, verdict={(Math.Abs(diff)<1e-9?"indifférence":diff>0?"déviation profitable":"coopération soutenable")}");}

Seuil théorique : 0.5000, horizon de troncature 500


delta=0.40, coopération=5.000, déviation=5.667, verdict=déviation profitable


delta=0.49, coopération=5.882, déviation=5.961, verdict=déviation profitable


delta=0.50, coopération=6.000, déviation=6.000, verdict=indifférence


delta=0.51, coopération=6.122, déviation=6.041, verdict=coopération soutenable


delta=0.60, coopération=7.500, déviation=6.500, verdict=coopération soutenable


Le seuil théorique est $\delta^\star = 0{,}5$ et la vérification numérique s'y colle exactement : à $\delta = 0{,}40$ et $0{,}49$ la déviation rapporte davantage (l'écart se referme quand $\delta$ monte), à $\delta = 0{,}51$ et $\delta = 0{,}60$ la coopération devient la meilleure réponse, et à $\delta = 0{,}50$ l'écart passe sous $10^{-9}$ — le verdict rendu est *indifférence au seuil*, ce qui est exactement la définition de $\delta^\star$ : le point où dévier ne rapporte plus rien mais ne coûte rien encore. C'est la version quantitative du **théorème folk** pour notre matrice : n'importe quel gain faisable individuellement rationnel — y compris $R$ — devient un équilibre pour un joueur assez patient (la démonstration formelle vit en Lean en [Dynamics-02](Dynamics/Dynamics-02-Jeux-Repetes-Lean-Python.ipynb), l'argument géométrique en [Dynamics-01](Dynamics/Dynamics-01-Jeux-Repetes-Folk-Theorem-Python.ipynb)). La coopération ne demande pas de bonté, seulement de la patience : les deux joueurs doivent accorder suffisamment de poids au futur.

### Exercice 1 — Une autre matrice

Pour $T=4$, $R=2$, $P=1$, $S=0$, calculez le seuil de patience et comparez-le au seuil de la section 3. Vérifiez-le ensuite par des sommes escomptées de part et d'autre.

Retournez aussi l'écart entre déviation et coopération pour les gains tronqués à 500 tours ; le garde contrôle les valeurs de part et d'autre du seuil.

In [4]:
Func<double,double,double,double?> seuilPatience=(t,r,p)=>{
 // TODO : seuil de patience
 return null;
};
Func<double,double,double,double,double?> ecartMatrice=(delta,t,r,p)=>{
 // TODO : déviation moins coopération sur 500 tours
 return null;
};
var casPatience=new[]{(5.0,3.0,1.0),(6.0,4.0,2.0),(8.0,5.0,1.0),(4.0,2.0,1.0)};
var repPatience=casPatience.Select(c=>seuilPatience(c.Item1,c.Item2,c.Item3)).ToArray();
var deltasExo=new[]{0.3,0.6,0.8};var ecartsExo=deltasExo.Select(d=>ecartMatrice(d,4,2,1)).ToArray();
if(repPatience.Any(x=>x is null)||ecartsExo.Any(x=>x is null))Console.WriteLine("Exercice 1 à compléter");
else{for(int i=0;i<casPatience.Length;i++){double x=repPatience[i].Value;if(!double.IsFinite(x)||x<0||x>=1||Math.Abs((casPatience[i].Item1-casPatience[i].Item2)-x*(casPatience[i].Item1-casPatience[i].Item3))>1e-9)throw new Exception("Sonde incorrecte");}for(int i=0;i<deltasExo.Length;i++){double d=deltasExo[i],x=ecartsExo[i].Value;double attendu=ValeurEscomptee(new[]{4.0}.Concat(Enumerable.Repeat(1.0,horizon-1)),d)-ValeurEscomptee(Enumerable.Repeat(2.0,horizon),d);if(!double.IsFinite(x)||Math.Abs(x-attendu)>1e-9)throw new Exception("Écart incorrect");} Console.WriteLine($"Exercice 1 : sondes validées ; autre matrice : {seuilPatience(4,2,1)}, écarts : {string.Join(", ",ecartsExo)}");}

Exercice 1 à compléter


## 4. Le moteur du tournoi

Une stratégie est une fonction qui, à chaque tour, regarde les deux historiques (le sien et celui de l'adversaire) et rend 'C' ou 'D'. Le moteur joue un match en N tours, avec un **bruit d'exécution** optionnel : avec probabilité p, l'action effectivement jouée est l'inverse de celle choisie — c'est le bruit le plus défavorable à la réciprocité, car une erreur de transmission est vue par l'adversaire comme une trahison.

In [5]:
Random rngGlobal=new Random(0);
Func<List<char>,List<char>,char> toujoursC=(m,a)=>'C',toujoursD=(m,a)=>'D',aleatoire=(m,a)=>rngGlobal.NextDouble()<0.5?'C':'D',tft=(m,a)=>a.Count==0?'C':a[^1];
Func<List<char>,List<char>,char> gtft=(m,a)=>a.Count==0||a[^1]=='C'?'C':rngGlobal.NextDouble()<1.0/3?'C':'D';
Func<List<char>,List<char>,char> rancunier=(m,a)=>a.Contains('D')?'D':'C';
Func<List<char>,List<char>,char> pavlov=(m,a)=>{if(a.Count==0)return 'C';var g=Gain(m[^1],a[^1]).Item1;return g==R||g==T?m[^1]:m[^1]=='C'?'D':'C';};
Func<List<char>,List<char>,char> sondeur=(m,a)=>m.Count<3?"DCC"[m.Count]:(a[1]=='C'&&a[2]=='C'?'D':a[^1]);
var strategies=new Dictionary<string,Func<List<char>,List<char>,char>>{{"ToujoursC",toujoursC},{"ToujoursD",toujoursD},{"Aleatoire",aleatoire},{"TitForTat",tft},{"GenereuxTFT",gtft},{"Rancunier",rancunier},{"Pavlov",pavlov},{"Sondeur",sondeur}};
(int,int) JouerMatch(Func<List<char>,List<char>,char> s1,Func<List<char>,List<char>,char> s2,int tours,double bruit=0,Random rng=null){
 var h1=new List<char>();var h2=new List<char>();int score1=0,score2=0;
 for(int i=0;i<tours;i++){char a=s1(h1,h2),b=s2(h2,h1);if(bruit>0&&rng!=null){if(rng.NextDouble()<bruit)a=a=='C'?'D':'C';if(rng.NextDouble()<bruit)b=b=='C'?'D':'C';}var g=Gain(a,b);score1+=g.Item1;score2+=g.Item2;h1.Add(a);h2.Add(b);}return(score1,score2);
}
Console.WriteLine($"Moteur : TitForTat contre ToujoursD, 5 tours = {JouerMatch(tft,toujoursD,5)} (attendu 4,9)");

Moteur : TitForTat contre ToujoursD, 5 tours = (4, 9) (attendu 4,9)


Le contrôle du moteur donne exactement les valeurs attendues : Tit-for-Tat, qui commence par coopérer puis copie, subit la dupe ($S=0$) au premier tour puis se résigne à la punition mutuelle ($P=1$ pendant quatre tours), pour un total de $0 + 4 = 4$ ; Toujours-D récolte la tentation puis la même punition, $5 + 4 \times 1 = 9$. Dans cet exemple bilatéral Toujours-D gagne — mais le tournoi ne classe pas les vainqueurs de matchs : il classe les totaux. Le moteur est prêt.

### Exercice 2 — La stratégie majoritaire

Implémentez `MajoriteC` : elle coopère si au moins la moitié des coups adverses sont C, y compris à égalité et au premier tour. Ajoutez-la ensuite au tournoi et comparez son rang à Rancunier et Généreux-TFT.

In [6]:
Func<List<char>,List<char>,char?> majoriteC=(moi,autre)=>{
 // TODO : choix selon l'historique adverse
 return null;
};
var historiques=new[]{"","CCD","CD","DD","CDD"};var attendusMajorite=new[]{'C','C','C','D','D'};
var repMajorite=historiques.Select(s=>majoriteC(new List<char>(),s.ToList())).ToArray();
if(repMajorite.Any(x=>x is null))Console.WriteLine("Exercice 2 à compléter");
else{if(!repMajorite.Select(x=>x.Value).SequenceEqual(attendusMajorite))throw new Exception("Sonde incorrecte");Console.WriteLine("Exercice 2 : sondes validées ; soumettez la stratégie au tournoi");}

Exercice 2 à compléter


## 5. Le tournoi sans bruit

Round-robin complet : chaque paire de stratégies se rencontre dans les **deux ordres** (les stratégies à mémoire d'état comme Sondeur ne sont pas symétriques), $N=200$ tours par match, graine fixée. Le classement se lit en **score moyen par tour** — une métrique indépendante du nombre de tours.

In [7]:
int nTours=200;
Dictionary<string,double> Tournoi(double bruit=0,int graine=0,int tours=200){
 rngGlobal=new Random(graine);var noms=strategies.Keys.ToArray();var scores=noms.ToDictionary(x=>x,x=>0.0);
 for(int i=0;i<noms.Length;i++)for(int j=i;j<noms.Length;j++){string a=noms[i],b=noms[j];if(a==b){var (s,_)=JouerMatch(strategies[a],strategies[b],tours,bruit,rngGlobal);scores[a]+=2*s;}else{var (sa,sb)=JouerMatch(strategies[a],strategies[b],tours,bruit,rngGlobal);var (sb2,sa2)=JouerMatch(strategies[b],strategies[a],tours,bruit,rngGlobal);scores[a]+=sa+sa2;scores[b]+=sb+sb2;}}
 return scores.ToDictionary(x=>x.Key,x=>x.Value/(2*noms.Length*tours));
}
var sansBruit=Tournoi();int rang=0;foreach(var item in sansBruit.OrderByDescending(x=>x.Value))Console.WriteLine($"{++rang} {item.Key,-12} {item.Value:F4}");

1 TitForTat    2.6566


2 GenereuxTFT  2.5831


3 Rancunier    2.5050


4 Pavlov       2.4659


5 Sondeur      2.2728


6 ToujoursD    2.1938


7 Aleatoire    2.1162


8 ToujoursC    2.0513


### Lecture

Le classement additionne les scores contre tous les adversaires : il ne compte pas simplement les victoires en duel. Avec la graine 0, Tit-for-Tat obtient 2,6566 en tête ; contre 2,1938 pour Toujours-D et 2,0513 pour Toujours-C. Les tirages .NET diffèrent de NumPy.

## 6. Le bruit d'exécution : la fragilité mesurée

Rejouons le tournoi avec un bruit $p \in \{0{,}02, 0{,}05, 0{,}10\}$ sur l'action effectivement jouée. Pour Tit-for-Tat, ce bruit est le pire scénario : une coopération transmise comme défection déclenche un **cycle d'échanges de punitions** — deux Tit-for-Tat peuvent s'échanger des défections alternées indéfiniment après une seule erreur. Quatre graines ($0, 1, 7, 42$) par niveau de bruit : le classement est une moyenne, l'écart-type entre graines est affiché.

In [8]:
var niveaux=new[]{0.0,0.02,0.05,0.10};var graines=new[]{0,1,7,42};
var table=new Dictionary<string,Dictionary<double,(double moyenne,double ecart)>>();
foreach(string nom in strategies.Keys)table[nom]=new Dictionary<double,(double,double)>();
foreach(double bruit in niveaux)foreach(string nom in strategies.Keys){var essais=graines.Select(g=>Tournoi(bruit,g)[nom]).ToArray();double moyenne=essais.Average(), ecart=Math.Sqrt(essais.Select(x=>(x-moyenne)*(x-moyenne)).Average());table[nom][bruit]=(moyenne,ecart);}
Console.WriteLine("Scores moyens et écarts-types, 4 graines par stratégie et niveau :");
foreach(string nom in strategies.Keys.OrderByDescending(n=>table[n][0].moyenne)){Console.Write($"{nom,-12}");foreach(double b in niveaux)Console.Write($" {table[nom][b].moyenne:F3} (+-{table[nom][b].ecart:F3})");Console.WriteLine();}
foreach(double b in niveaux){var classement=strategies.Keys.OrderByDescending(n=>table[n][b].moyenne).ToList();Console.WriteLine($"bruit={b:F2}: rang TitForTat={classement.IndexOf("TitForTat")+1}, premier={classement[0]}");}

Scores moyens et écarts-types, 4 graines par stratégie et niveau :


TitForTat   

 2.652 (+-0.003)

 2.232 (+-0.082)

 2.125 (+-0.081)

 2.151 (+-0.041)

... (35 outputs left out)

bruit=0.00: rang TitForTat=1, premier=TitForTat


bruit=0.02: rang TitForTat=3, premier=Rancunier


bruit=0.05: rang TitForTat=5, premier=Rancunier


bruit=0.10: rang TitForTat=5, premier=Rancunier


### Lecture

Avec les graines .NET, Tit-for-Tat passe de **2,652** sans bruit à **2,151** à $p=0{,}10$, et de la première à la cinquième place. À $p=0{,}02$, il reste troisième ; Rancunier est premier. Une erreur peut faire alterner les représailles de Tit-for-Tat. Les tirages .NET diffèrent de NumPy ; les rangs et écarts observés sont propres à ces graines.

### Exercice 3 — Le seuil de bruit où Tit-for-Tat perd sa place

Balayez le bruit entre 0 et 0,15 par pas donné ; pour chaque valeur, calculez la moyenne des scores de chaque stratégie sur les graines fournies et retournez le premier bruit où Tit-for-Tat sort du podium. Le paramètre `calcul_tournoi` permet de vérifier la recherche sur des tournois fictifs sans remplacer la simulation réelle.

In [9]:
Func<double,int[],Func<double,int,Dictionary<string,double>>,double?> seuilPertePodium=(pas,grainesExo,calculTournoi)=>{
 // TODO : premier bruit où le rang moyen dépasse 3
 return null;
};
Func<double,Func<double,int,Dictionary<string,double>>> fixtureSeuil=seuil=>(bruit,graine)=>new Dictionary<string,double>{{"TitForTat",bruit<seuil-1e-9?4:0},{"A",3},{"B",2},{"C",1}};
var repSeuil=new[]{seuilPertePodium(0.01,graines,fixtureSeuil(0.03)),seuilPertePodium(0.02,graines,fixtureSeuil(0.08))};
if(repSeuil.Any(x=>x is null))Console.WriteLine("Exercice 3 à compléter");
else{if(!double.IsFinite(repSeuil[0].Value)||!double.IsFinite(repSeuil[1].Value)||Math.Abs(repSeuil[0].Value-0.03)>1e-9||Math.Abs(repSeuil[1].Value-0.08)>1e-9)throw new Exception("Sonde incorrecte");var reel=seuilPertePodium(0.01,graines,(b,g)=>Tournoi(b,g)); if(reel is null || !double.IsFinite(reel.Value) || reel.Value<0 || reel.Value>0.15 || Math.Abs(reel.Value*100-Math.Round(reel.Value*100))>1e-8) throw new Exception("Seuil réel invalide");
 Func<double,int> rangReel=b=>{var essais=graines.Select(g=>Tournoi(b,g)).ToArray();var classement=strategies.Keys.OrderByDescending(n=>essais.Average(e=>e[n])).ToList();return classement.IndexOf("TitForTat")+1;};
 if(rangReel(reel.Value)<=3 || Enumerable.Range(0,(int)Math.Round(reel.Value*100)).Any(k=>rangReel(k/100.0)>3))throw new Exception("Seuil réel incorrect"); Console.WriteLine($"Exercice 3 : sondes validées ; seuil simulé : {reel}");}

Exercice 3 à compléter


## Synthèse

Trois mécanismes séparent le dilemme répété de l'épisode unique, et chacun a été **mesuré** ici :

- **Horizon fini** : la récursion arrière dénoue la coopération tour après tour — cinq horizons testés, l'équilibre au tour 1 est toujours $(D, D)$.
- **Horizon infini** : la patience rend la menace crédible — seuil exact $\delta^\star = (T-R)/(T-P) = 0{,}5$ pour notre matrice, vérifié de part et d'autre par valeurs escomptées tronquées, avec l'indifférence exacte au seuil.
- **Tournoi** : la réciprocité gagne en environnement propre et s'effondre en environnement bruité — le tableau indique les rangs et scores obtenus avec les graines .NET : comparer les réalisations et leur dispersion.

**Ce qu'on emporte** : la coopération n'est pas un trait de caractère mais une propriété de la structure du jeu — longueur d'horizon, patience, fiabilité du canal. Changer un de ces trois paramètres suffit à faire basculer l'équilibre, dans un sens ou dans l'autre.

**Note bibliographique** : ce carnet est auto-contenu et n'attribue aucun résultat historique à une publication — toutes les valeurs affichées proviennent de la simulation locale. L'ancrage historique des tournois de jeux répétés (Axelrod, 1980-1984) attend le dépôt de la copie de référence au gisement bibliographique partagé ; il sera ajouté à cette section au moment du dépôt.

## Exercices — récapitulatif

Les exercices répartis dans les sections 3, 4 et 6 portent sur le seuil de patience, la stratégie majoritaire et la recherche du seuil de bruit.

## À retenir

Un horizon fini connu produit la défection par récurrence ; l'escompte peut soutenir la coopération ; le bruit change les scores d'un tournoi de stratégies.

## Vérifiez votre compréhension

1. Pourquoi le dernier tour joue-t-il un rôle dans l'horizon fini ?

<details><summary>Réponse</summary>Il ne reste aucune continuation à protéger ; la défection domine, et le raisonnement remonte les tours.</details>

2. Le rang d'une stratégie est-il garanti sous une autre graine ?

<details><summary>Réponse</summary>Non, les décisions aléatoires et erreurs de transmission changent les scores ; comparer plusieurs graines réduit la dépendance à une réalisation sans supprimer l'incertitude.</details>

## Pour aller plus loin

- [Jeux répétés et théorème folk](Dynamics/Dynamics-01-Jeux-Repetes-Folk-Theorem-Python.ipynb).
- [Évolution de la confiance](GameTheory-06-EvolutionTrust-Python.ipynb).

## Lectures

- Osborne & Rubinstein, *A Course in Game Theory* (MIT Press, 1994), ch. 8 « Repeated Games » : § 8.2 « Infinitely Repeated Games vs. Finitely Repeated Games » (p. 134) pour l'opposition que le carnet mesure en premier, § 8.5 « Trigger Strategies: Nash Folk Theorems » (p. 143) pour la crédibilité de la menace que le seuil $\delta^\star$ rend exacte, et § 8.10 « Finitely Repeated Games » (p. 155) pour la récursion arrière qui dénoue l'horizon fini.